# Topic Modeling

Discover hidden topics in the reviews using:
- LDA (Latent Dirichlet Allocation) — traditional approach
- BERTopic — transformer-based approach

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

from config import TRAIN_CSV, RANDOM_STATE

train = pd.read_csv(TRAIN_CSV)
train["review_heavy"] = train["review_heavy"].fillna("")

print(f"Train: {len(train)} reviews")

Train: 6226 reviews


## 1. LDA Topic Modeling

In [3]:
# Tokenize for LDA
texts = train["review_heavy"].str.split().tolist()

# Build dictionary and corpus
dictionary = corpora.Dictionary(texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in texts]

print(f"Dictionary size: {len(dictionary)} unique tokens")
print(f"Corpus size: {len(corpus)} documents")

Dictionary size: 4700 unique tokens
Corpus size: 6226 documents


In [4]:
# Train LDA model with 5 topics
num_topics = 5

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=RANDOM_STATE,
    passes=10,
    alpha="auto",
    per_word_topics=True
)

# Display topics
print("LDA Topics:\n")
for idx, topic in lda_model.print_topics(num_words=8):
    print(f"Topic {idx}: {topic}\n")

LDA Topics:

Topic 0: 0.023*"play" + 0.017*"war" + 0.017*"product" + 0.017*"fun" + 0.016*"free" + 0.014*"dlc" + 0.014*"like" + 0.013*"card"

Topic 1: 0.010*"like" + 0.009*"get" + 0.008*"time" + 0.007*"would" + 0.007*"play" + 0.007*"one" + 0.007*"even" + 0.006*"make"

Topic 2: 0.013*"like" + 0.010*"character" + 0.009*"feel" + 0.009*"one" + 0.007*"story" + 0.007*"get" + 0.007*"world" + 0.007*"time"

Topic 3: 0.031*"good" + 0.017*"grind" + 0.013*"bad" + 0.011*"money" + 0.011*"car" + 0.011*"get" + 0.011*"bug" + 0.010*"hard"

Topic 4: 0.018*"fun" + 0.017*"good" + 0.017*"like" + 0.015*"get" + 0.015*"kill" + 0.015*"great" + 0.014*"really" + 0.014*"die"



In [5]:
# Evaluate LDA with coherence score
coherence_model = CoherenceModel(
    model=lda_model,
    texts=texts,
    dictionary=dictionary,
    coherence="c_v"
)

coherence_score = coherence_model.get_coherence()
print(f"Coherence Score (c_v): {coherence_score:.4f}")
print(f"\n(Higher is better. Typical range: 0.3-0.7)")

Coherence Score (c_v): 0.3705

(Higher is better. Typical range: 0.3-0.7)


## 2. BERTopic

In [8]:
from bertopic import BERTopic
from umap import UMAP
from src.embeddings import encode_sentences
from config import SENTENCE_MODEL_NAME, RANDOM_STATE

# Encode training reviews with Sentence Transformer
_, train_st_embeddings = encode_sentences(train["review_clean"], model_name=SENTENCE_MODEL_NAME)

# Fit BERTopic (random_state goes through UMAP)
umap_model = UMAP(random_state=RANDOM_STATE)
topic_model = BERTopic(nr_topics=5, umap_model=umap_model)
topics, probs = topic_model.fit_transform(train["review_clean"].tolist(), train_st_embeddings)

# Show topics
print("BERTopic Topics:\n")
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/98 [00:00<?, ?it/s]

BERTopic Topics:

   Topic  Count                            Name  \
0     -1   2432                -1_the_and_to_is   
1      0   3627                 0_the_and_to_is   
2      1     85           1_good_alright_it_its   
3      2     64                  2_de_la_que_nk   
4      3     18  3_wallpapers_wallpaper_app_you   

                                      Representation  \
0    [the, and, to, is, game, of, it, you, this, in]   
1    [the, and, to, is, game, it, of, you, this, in]   
2  [good, alright, it, its, gay, very, is, guess,...   
3         [de, la, que, nk, aku, ni, et, da, ko, tu]   
4  [wallpapers, wallpaper, app, you, is, and, are...   

                                 Representative_Docs  
0  [edit for a couple of additional thoughts afte...  
1  [i think recorded steam hours plus countless m...  
2               [it is good, it is good, it is good]  
3  [sendo um sucesso dentre os jogos rogue lite s...  
4  [product received for free early access review...  


## Topic Modeling Summary

| Method | Coherence / Quality | Strengths | Weaknesses |
|--------|-------------------|-----------|------------|
| LDA | c_v = 0.37 | Interpretable topics, fast | High word overlap between topics |
| BERTopic | N/A (cluster-based) | Finds niche topics, uses context | Many outliers (-1), generic main topic |

Key findings:
- Both methods struggle with gaming reviews due to shared vocabulary across topics
- LDA found broad themes: F2P/DLC, narrative, complaints, action
- BERTopic identified niche clusters but grouped most reviews into one large topic
- This confirms that the main signal in these reviews is sentiment (positive/negative), not topic